In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import importlib


In [3]:
import copy
import sys
import os

sys.path.append(os.path.abspath("../"))
from campaign_diagram import *
# import campaign_diagram
# campaign_diagram.__file__
# campaign_diagram.CampaignDiagram
## IN this notebook
# guard_position, dual mode, etc....

#### Sometimes, there is more than one type of compute (or memory) resource. How do we address this? (IPR)

- We indicate resource type by hash patterns
- However, ALL resources will be grouped into compute type or memory type.
- the Y axis is the SUM of all resources being fully utilized.
- this approach ONLY works with the dual_mode, it will not work with the colocated mode.
- each bar will have a solid line for th etime it would have taken to achieve the particular resource if at 100% utilization, and a dashed line for the actual time it took
- this seems like a deviation from what we have so far?


## Three-Stage Pipeline Again

In [4]:
# kernel3a = Kernel(name='EinsumA',
#                   start=0,
#                   duration=5,
#                   compute_util=0.8,
#                   bw_util=0.15)


# kernel3b = Kernel(name='EinsumB',
#                   start=kernel3a.end,
#                   duration=10,
#                   compute_util=0.2,
#                   bw_util=0.6)


# kernel3c = Kernel(name='EinsumC',
#                   start=kernel3b.end,
#                   duration=5,
#                   compute_util=0.5,   # was 0.2
#                   bw_util=0.25)

kernel3a = Kernel(name='EinsumA',
                  start=0,
                  duration=5,
                  compute_util=0.2,
                  bw_util=1)


kernel3b = Kernel(name='EinsumB',
                  start=kernel3a.end,
                  duration=10,
                  compute_util=0.3,
                  bw_util=0.6)


kernel3c = Kernel(name='EinsumC',
                  start=kernel3b.end,
                  duration=5,
                  compute_util=0.8,   # was 0.2
                  bw_util=0.25)

original_cascade = Cascade(name="Three-step Cascade",
                           kernels=[kernel3a, kernel3b, kernel3c])

In [5]:
campaign_diagram = CampaignDiagram(original_cascade)
CampaignDiagram(original_cascade).draw(alpha=0.5).interactive().show()
CampaignDiagram(original_cascade).draw(dual_mode=True, alpha=0.5).interactive()

alt.LayerChart(...)

alt.LayerChart(...)

### What happens when we pipeline?

- Suppose we now pipeline the first two kernels.
- If a kernel had ALL the available resources, the solid line should show that runtime.
- The dashed line shows the remainder runtime due to an unideal kernel or due to needing to allow other kernels to run concurrently.



In [6]:
num_of_tiles = 4
tiled_cascade = original_cascade.tile(num_of_tiles)
campaign_diagram = CampaignDiagram(tiled_cascade)
campaign_diagram.draw().interactive().show()
campaign_diagram.draw(dual_mode=True).interactive()

alt.LayerChart(...)

alt.LayerChart(...)

In [7]:
pipelined_cascade = tiled_cascade.pipeline(3)

campaign_diagram = CampaignDiagram(pipelined_cascade)
campaign_diagram.draw().interactive().show()
campaign_diagram.draw(dual_mode=True).interactive()

alt.LayerChart(...)

alt.LayerChart(...)

In [8]:
throttled_cascade = tiled_cascade.pipeline(3).throttle()

campaign_diagram = CampaignDiagram(throttled_cascade)
campaign_diagram.draw().interactive().show()
campaign_diagram.draw(dual_mode=True).interactive()

alt.LayerChart(...)

alt.LayerChart(...)

In [9]:
dilated_cascade = tiled_cascade.pipeline(stages=3, spread=True)

campaign_diagram = CampaignDiagram(dilated_cascade)
campaign_diagram.draw().interactive().show()
campaign_diagram.draw(dual_mode=True).interactive()

alt.LayerChart(...)

alt.LayerChart(...)

In [10]:
dilated_cascade = tiled_cascade.pipeline(stages=3, spread=True)

campaign_diagram = CampaignDiagram(dilated_cascade.throttle())
campaign_diagram.draw().interactive().show()
campaign_diagram.draw(dual_mode=True).interactive()

alt.LayerChart(...)

alt.LayerChart(...)

## Setup Resources
- Let Kernel4a use some special compute resource (let's say a 1D array with special functions)
- Let Kernel 4b use some other compute resource (let's say a 2D array that only it can use)
- Let Kernel 4a and 4b both use the same generic memory (DRAM)
- Let Kernel 4c use some special memory resource and the 2D array.
- Assume C depends on B depends on A.

### Accumulative vs. non-accumulative resources

Every sub-resource carries an **`accumulative`** flag that decides how its band
**height** (`p_of_tot`) is computed. `derive_p_of_tot()` reads the flag:

- **Compute is accumulative** (`add_compute(..., accumulative=True)`, the
  default). Compute sub-resources **pool** onto one FLOP/s axis, so their
  heights are **peak-shares** that *sum* to the whole compute pool
  (`peak[r] / Σ peaks`, or an explicit hand-set share). Stacking them is
  meaningful: two compute engines running at once really do draw from one
  combined budget.

- **Memory is non-accumulative** (`add_memory(..., accumulative=False)`, the
  default). Memory levels form a **hierarchy** (L2, DRAM, ...), not a shared
  pool: the *same* bytes that a kernel touches in L2 are the bytes streamed from
  DRAM, so the utilizations **overlap** and adding them is meaningless. Instead
  the memory axis is split into **equal `1/N` slots** (ignoring peak ratios),
  and each slot is read as an **independent 0..1 gauge** — fill / slot-height is
  that one level's own utilization. The slots stack only for layout and are
  **never summed**; an unbound level still keeps (and draws) its empty slot.

The cells below keep the compute peak-shares but let memory take equal slots
(FancyMem = DRAM = 0.5) via `reg.derive_p_of_tot()`. A separate worked example
at the very bottom shows the compute **peak-share** path (Tensor vs FMA) beside
the memory equal-slot path, including an empty (unbound) memory slot.

In [11]:
# A campaign diagram is 
# kernel4a = Kernel(name='EinsumA',
#                   start=0,
#                   duration=5,
#                   compute_util=1,
#                   bw_util=0.15, 
#                   compute_type = "Special1D",
#                   memory_type = "DRAM"
#                  )


# kernel4b = Kernel(name='EinsumB',
#                   start=kernel4a.end,
#                   duration=10,
#                   compute_util=0.2,
#                   bw_util=1,
#                   compute_type = "Special2D",
#                   memory_type = "DRAM"
#                  )


# kernel4c = Kernel(name='EinsumC',
#                   start=kernel4b.end,
#                   duration=5,
#                   compute_util=0.5,   # was 0.2
#                   bw_util=0.25,
#                   compute_type = "Special2D",
#                   memory_type = "FancyMem"
#                  )

# A campaign diagram is 
kernel4a = Kernel(name='EinsumA',
                  start=0,
                  duration=5,
                  compute_util=0.3,
                  bw_util=0.15, 
                  compute_type = "Special1D",
                  memory_type = "DRAM"
                 )


kernel4b = Kernel(name='EinsumB',
                  start=kernel4a.end,
                  duration=10,
                  compute_util=0.2,
                  bw_util=0.6,
                  compute_type = "Special2D",
                  memory_type = "DRAM"
                 )


kernel4c = Kernel(name='EinsumC',
                  start=kernel4b.end,
                  duration=5,
                  compute_util=0.5,   # was 0.2
                  bw_util=0.25,
                  compute_type = "Special2D",
                  memory_type = "DRAM"
                 )


In [ ]:
## Now define our resources/set it up, using the CORRECTED data-model API.
#
# The previous version of this cell was broken in two ways:
#   1. It called `bind.get_mem_resource_name(k)` / `bind.get_comp_resource_name(k)`
#      with the OLD signature expectations (the binder methods were ambiguous /
#      one shadowed the other), and
#   2. More importantly, it relied on the OLD behaviour where
#      `update_global_util_view` clobbered `compute_type` / `memory_type` and
#      mis-scaled the utilizations.
#
# The corrected model is much simpler: every Kernel already carries its chosen
# sub-resource NAME (compute_type / memory_type, set in the constructor). All we
# have to do is (a) register every sub-resource name in a ResourceRegistry with
# its pool weight `p_of_tot`, then (b) ask each kernel to recompute its
# pool-relative view via `k.update_global_util_view(registry=reg)`. The kernel
# resolves its own weight from the registry by name. No ResourceBinder needed.

################################################################################
# STEP 1: (Re)build the three kernels for the multi-resource scenario.
#
# We rebuild them here so this cell is self-contained and the data is
# unambiguous (the earlier kernel-definition cell mistakenly gave EinsumC a
# "DRAM" memory_type; the multi-resource design intends "FancyMem").
#
#   EinsumA : duration 5, compute_util 0.30, bw_util 0.15,
#             compute on "Special1D", memory on "DRAM"
#   EinsumB : duration 10, compute_util 0.20, bw_util 0.60,
#             compute on "Special2D", memory on "DRAM"
#   EinsumC : duration 5, compute_util 0.50, bw_util 0.25,
#             compute on "Special2D", memory on "FancyMem"
#
# compute_util / bw_util passed to the constructor are the PER-SUB-RESOURCE
# (raw) utilizations -- i.e. "what fraction of THAT 1D / 2D array (or that
# memory) does this kernel use". They become compute_util_raw / bw_util_raw,
# which are the immutable source of truth.
################################################################################

# EinsumA runs on the special 1D compute array and the generic DRAM memory.
kernel4a = Kernel(name='EinsumA',
                  start=0,
                  duration=5,
                  compute_util=0.3,      # 30% of the Special1D array
                  bw_util=0.15,          # 15% of DRAM bandwidth
                  compute_type="Special1D",
                  memory_type="DRAM")

# EinsumB runs on the special 2D compute array and the same generic DRAM.
kernel4b = Kernel(name='EinsumB',
                  start=kernel4a.end,    # C depends on B depends on A: chain them
                  duration=10,
                  compute_util=0.2,      # 20% of the Special2D array
                  bw_util=0.6,           # 60% of DRAM bandwidth
                  compute_type="Special2D",
                  memory_type="DRAM")

# EinsumC also runs on the special 2D compute array, but uses a special
# memory ("FancyMem") instead of DRAM.
kernel4c = Kernel(name='EinsumC',
                  start=kernel4b.end,
                  duration=5,
                  compute_util=0.5,      # 50% of the Special2D array
                  bw_util=0.25,          # 25% of FancyMem bandwidth
                  compute_type="Special2D",
                  memory_type="FancyMem")

# Keep them in a list so the steps below can iterate uniformly.
kernels = [kernel4a, kernel4b, kernel4c]

################################################################################
# STEP 2: Register every sub-resource NAME with its pool weight `p_of_tot`.
#
# `p_of_tot` is the fixed BAND HEIGHT for each sub-resource, and the
# `accumulative` flag decides how it is computed:
#   * ACCUMULATIVE compute (the add_compute default): the sub-resources POOL
#     onto one FLOP/s axis, so `p_of_tot` is the fraction of the WHOLE compute
#     pool -- Special1D = 25%, Special2D = 75%, summing to 1.0.
#   * NON-ACCUMULATIVE memory (the add_memory default): memory levels form a
#     HIERARCHY (a byte "used" at one level is the same byte that moved through
#     the next), so summing their utilizations is meaningless. Instead the axis
#     is split into EQUAL 1/N slots -- FancyMem = DRAM = 0.5 -- and each slot is
#     read as an INDEPENDENT 0..1 gauge, never summed. `derive_p_of_tot()` sets
#     these heights from the flags.
#
# IMPORTANT: a sub-resource name MUST be registered before any kernel that
# names it calls update_global_util_view(registry=reg) -- registry.get() raises
# KeyError for an unknown name. (The registry also auto-registers the
# single-resource defaults "Compute Pool" and "Memory Pool".)
#
# DECLARATION ORDER MATTERS FOR THE RENDER: when this `reg` is later handed to
# CampaignDiagram(..., resource_registry=reg) for the multi-resource view, the
# stacked resource bands are laid out in THIS declaration order -- NOT in
# kernel-appearance order. The user's chosen order deliberately registers
# FancyMem BEFORE DRAM, so on the (mirrored) memory axis FancyMem is the slot
# nearest zero ([0 -> -0.5]) and DRAM is the equal slot beyond it ([-0.5 -> -1.0]).
# Similarly Special1D BEFORE Special2D puts Special1D nearest zero on the
# compute axis ([0 -> 0.25]) and Special2D beyond it ([0.25 -> 1.0]).
################################################################################

reg = ResourceRegistry()
# Compute stays ACCUMULATIVE (the add_compute default). We hand-set explicit
# pool shares (no `peak`), which derive_p_of_tot() leaves untouched -> 0.25/0.75.
reg.add_compute("Special1D", p_of_tot=0.25)   # 1D array = 25% of all compute
reg.add_compute("Special2D", p_of_tot=0.75)   # 2D array = 75% of all compute
# Memory is NON-ACCUMULATIVE (the add_memory default): no hand-set share -- the
# levels get EQUAL 1/N slots from derive_p_of_tot() below (here 0.5 each).
reg.add_memory("FancyMem")                     # equal slot -> 0.5 after derive
reg.add_memory("DRAM")                         # equal slot -> 0.5 after derive
# Set every band height from the accumulative flags: compute keeps its explicit
# peak-shares, memory becomes equal 1/N slots. Replaces the old 0.1/0.9.
reg.derive_p_of_tot()

################################################################################
# STEP 3: Ask each kernel to recompute its POOL-RELATIVE utilization view.
#
# This is the primary corrected API. The kernel reads its own compute_type /
# memory_type, looks each up in `reg`, pulls `p_of_tot`, and sets:
#     compute_util = compute_util_raw * (compute sub-resource p_of_tot)
#     bw_util      = bw_util_raw      * (memory  sub-resource p_of_tot)
# It also recomputes the ideal / throttled durations, which are RELATIVE TO
# THE KERNEL'S OWN SUB-RESOURCE (they use the _raw values, NOT the pool view).
#
# NOTE: these per-kernel numbers depend ONLY on each resource's own p_of_tot,
# so they are completely INDEPENDENT of the registry DECLARATION order above.
# Reordering add_memory(FancyMem) vs add_memory(DRAM) changes only the visual
# band stacking, never compute_util / bw_util / the durations -- which is why
# the data-model assertion cell below stays valid unchanged.
#
# The call is idempotent: every derived field is recomputed from the immutable
# _raw values, so calling it again with the same registry changes nothing.
# (A ResourceBinder is redundant under the corrected model and is intentionally
# not used here.)
################################################################################

for k in kernels:
    k.update_global_util_view(registry=reg)

################################################################################
# STEP 4: Assemble the cascade the rest of the notebook expects.
################################################################################

multi_r_cascade = Cascade(name="Three-step Cascade",
                          kernels=[kernel4a, kernel4b, kernel4c])

# Quick human-readable echo so we can eyeball the result immediately.
for k in kernels:
    print(f"{k.name}: compute_type={k.compute_type} "
          f"compute_util_raw={k.compute_util_raw} -> compute_util={k.compute_util:.6g} "
          f"| memory_type={k.memory_type} "
          f"bw_util_raw={k.bw_util_raw} -> bw_util={k.bw_util:.6g}")


### Expected numbers (data-model proof)

Utilization drawn on a band = `*_raw * p_of_tot`. For ACCUMULATIVE compute the
`p_of_tot` is the pool share (0.25 / 0.75); for NON-ACCUMULATIVE memory it is
the equal 1/N slot (0.5 / 0.5), so the memory number reads as that level's own
independent gauge, not a share of a combined memory pool. Ideal durations are
**relative to the kernel's own sub-resource** and use the *raw* utilizations
(so they are unaffected by the memory height change).

| kernel | compute_type | compute_util (pool) | memory_type | bw_util (pool) |
|--------|--------------|---------------------|-------------|----------------|
| EinsumA | Special1D (p=0.25) | 0.3 * 0.25 = **0.075** | DRAM (p=0.5) | 0.15 * 0.5 = **0.075** |
| EinsumB | Special2D (p=0.75) | 0.2 * 0.75 = **0.15** | DRAM (p=0.5) | 0.6 * 0.5 = **0.30** |
| EinsumC | Special2D (p=0.75) | 0.5 * 0.75 = **0.375** | FancyMem (p=0.5) | 0.25 * 0.5 = **0.125** |

Ideal / throttled durations (sub-resource-relative, from the `_raw` values):

| kernel | duration | cb_ideal = dur*cu_raw | mb_ideal = dur*bw_raw | ideal = max | throttled = dur - ideal |
|--------|----------|-----------------------|-----------------------|-------------|--------------------------|
| EinsumA | 5  | 5*0.3 = **1.5**  | 5*0.15 = **0.75** | **1.5** | 5 - 1.5 = **3.5** |
| EinsumB | 10 | 10*0.2 = **2.0** | 10*0.6 = **6.0**  | **6.0** | 10 - 6.0 = **4.0** |
| EinsumC | 5  | 5*0.5 = **2.5**  | 5*0.25 = **1.25** | **2.5** | 5 - 2.5 = **2.5** |

`compute_type` / `memory_type` must remain **unchanged** by the call, and the
call must be **idempotent** (a second `update_global_util_view(registry=reg)`
yields identical values).

In [ ]:
import math

################################################################################
# DATA-MODEL ASSERTIONS
#
# We re-derive every expected value from first principles (raw utilization,
# pool weight, duration) and check the kernel objects match. `close()` is a
# tolerant float compare so harmless IEEE-754 noise (e.g. 0.2*0.75 ==
# 0.15000000000000002) does not cause a spurious failure. Any real mismatch
# raises AssertionError LOUDLY with both expected and actual values.
################################################################################

def close(actual, expected, label):
    """Assert actual ~= expected, reporting both numbers if it fails."""
    assert math.isclose(actual, expected, rel_tol=0.0, abs_tol=1e-9), (
        f"{label}: expected {expected!r} but got {actual!r}")

# Pool weights, mirrored here independently of the registry so the assertions
# are a genuine cross-check rather than a tautology.
P_SPECIAL1D = 0.25
P_SPECIAL2D = 0.75
P_DRAM      = 0.5   # equal 1/N memory slot (non-accumulative), N=2
P_FANCYMEM  = 0.5   # equal 1/N memory slot (non-accumulative), N=2

# expected[name] = (compute_type, memory_type, comp_p_of_tot, mem_p_of_tot,
#                   duration, compute_util_raw, bw_util_raw)
expected = {
    "EinsumA": ("Special1D", "DRAM",     P_SPECIAL1D, P_DRAM,     5,  0.3, 0.15),
    "EinsumB": ("Special2D", "DRAM",     P_SPECIAL2D, P_DRAM,     10, 0.2, 0.6),
    "EinsumC": ("Special2D", "FancyMem", P_SPECIAL2D, P_FANCYMEM, 5,  0.5, 0.25),
}

################################################################################
# Pass 1: validate the freshly-computed view for every kernel.
################################################################################

for k in kernels:
    (exp_ct, exp_mt, exp_cp, exp_mp,
     exp_dur, exp_cu_raw, exp_bw_raw) = expected[k.name]

    # --- sub-resource names must be UNCHANGED by update_global_util_view ---
    assert k.compute_type == exp_ct, (
        f"{k.name}: compute_type clobbered -> {k.compute_type!r} "
        f"(expected {exp_ct!r})")
    assert k.memory_type == exp_mt, (
        f"{k.name}: memory_type clobbered -> {k.memory_type!r} "
        f"(expected {exp_mt!r})")

    # --- raw utilizations are the immutable source of truth ---
    close(k.compute_util_raw, exp_cu_raw, f"{k.name} compute_util_raw")
    close(k.bw_util_raw,      exp_bw_raw, f"{k.name} bw_util_raw")

    # --- resolved pool weights surfaced for the tooltip layer ---
    close(k.comp_perc, exp_cp, f"{k.name} comp_perc")
    close(k.bw_perc,   exp_mp, f"{k.name} bw_perc")

    # --- pool-relative utilization = raw * p_of_tot ---
    close(k.compute_util, exp_cu_raw * exp_cp, f"{k.name} compute_util(pool)")
    close(k.bw_util,      exp_bw_raw * exp_mp, f"{k.name} bw_util(pool)")

    # --- ideal / throttled durations are SUB-RESOURCE-relative (use _raw) ---
    exp_cb_ideal = exp_dur * exp_cu_raw
    exp_mb_ideal = exp_dur * exp_bw_raw
    exp_ideal    = max(exp_cb_ideal, exp_mb_ideal)
    exp_throttled = exp_dur - exp_ideal

    close(k.cb_ideal_duration,  exp_cb_ideal,  f"{k.name} cb_ideal_duration")
    close(k.mb_ideal_duration,  exp_mb_ideal,  f"{k.name} mb_ideal_duration")
    close(k.ideal_duration,     exp_ideal,     f"{k.name} ideal_duration")
    close(k.throttled_duration, exp_throttled, f"{k.name} throttled_duration")

################################################################################
# Pass 2: IDEMPOTENCY -- calling update_global_util_view a second time with the
# same registry must not change any derived field.
################################################################################

for k in kernels:
    before = (k.compute_util, k.bw_util, k.comp_perc, k.bw_perc,
              k.cb_ideal_duration, k.mb_ideal_duration,
              k.ideal_duration, k.throttled_duration,
              k.compute_type, k.memory_type)

    k.update_global_util_view(registry=reg)   # second call, same registry

    after = (k.compute_util, k.bw_util, k.comp_perc, k.bw_perc,
             k.cb_ideal_duration, k.mb_ideal_duration,
             k.ideal_duration, k.throttled_duration,
             k.compute_type, k.memory_type)

    assert before == after, (
        f"{k.name}: update_global_util_view is NOT idempotent\n"
        f"  before={before}\n  after ={after}")

print("ALL MULTI-RESOURCE DATA-MODEL ASSERTIONS PASSED")


In [14]:
import pandas as pd

################################################################################
# DATA-PROOF TABLE
#
# One row per kernel showing the full derivation chain side by side:
# the raw (per-sub-resource) utilization, the pool weight it was scaled by,
# the resulting pool-relative utilization, and the sub-resource-relative
# ideal / throttled durations. Eyeball this against the "Expected numbers"
# markdown cell above.
################################################################################

rows = []
for k in kernels:
    rows.append({
        "kernel":             k.name,
        "compute_type":       k.compute_type,
        "compute_util_raw":   k.compute_util_raw,
        "comp p_of_tot":      k.comp_perc,
        "compute_util(pool)": k.compute_util,
        "memory_type":        k.memory_type,
        "bw_util_raw":        k.bw_util_raw,
        "mem p_of_tot":       k.bw_perc,
        "bw_util(pool)":      k.bw_util,
        "cb_ideal":           k.cb_ideal_duration,
        "mb_ideal":           k.mb_ideal_duration,
        "ideal":              k.ideal_duration,
        "throttled":          k.throttled_duration,
    })

multi_r_proof_df = pd.DataFrame(rows, columns=[
    "kernel", "compute_type", "compute_util_raw", "comp p_of_tot",
    "compute_util(pool)", "memory_type", "bw_util_raw", "mem p_of_tot",
    "bw_util(pool)", "cb_ideal", "mb_ideal", "ideal", "throttled",
])

print(multi_r_proof_df.to_string(index=False))
multi_r_proof_df


 kernel compute_type  compute_util_raw  comp p_of_tot  compute_util(pool) memory_type  bw_util_raw  mem p_of_tot  bw_util(pool)  cb_ideal  mb_ideal  ideal  throttled
EinsumA    Special1D               0.3           0.25               0.075        DRAM         0.15           0.9          0.135       1.5      0.75    1.5        3.5
EinsumB    Special2D               0.2           0.75               0.150        DRAM         0.60           0.9          0.540       2.0      6.00    6.0        4.0
EinsumC    Special2D               0.5           0.75               0.375    FancyMem         0.25           0.1          0.025       2.5      1.25    2.5        2.5


,kernel,compute_type,compute_util_raw,comp p_of_tot,compute_util(pool),memory_type,bw_util_raw,mem p_of_tot,bw_util(pool),cb_ideal,mb_ideal,ideal,throttled
0,EinsumA,Special1D,0.3,0.25,0.075,DRAM,0.15,0.9,0.135,1.5,0.75,1.5,3.5
1,EinsumB,Special2D,0.2,0.75,0.150,DRAM,0.60,0.9,0.540,2.0,6.00,6.0,4.0
2,EinsumC,Special2D,0.5,0.75,0.375,FancyMem,0.25,0.1,0.025,2.5,1.25,2.5,2.5


### Expected multi-resource visual (compare to `SCRIBBLES_2025.jpeg`)

The cell below renders the **opt-in multi-resource view** with
`CampaignDiagram(multi_r_cascade, resource_registry=reg)` followed by
`.draw(dual_mode=True, multi_resource=True)`. Passing `resource_registry=reg`
is what makes the stacked resource bands follow the **registry DECLARATION
order** (Special1D, Special2D, FancyMem, DRAM) instead of the legacy
kernel-appearance order. This visual ONLY works in **dual_mode** (compute
mirrored on +Y, memory on -Y) -- passing `multi_resource=True` also
auto-enables `dual_mode`, but we pass it explicitly for clarity.

#### Expected band layout (with the registry order Special1D, Special2D, FancyMem, DRAM)

Because we declared the resources in this exact order, the stacked bands are:

- **Compute axis (+Y), stacked from zero outward:**
  - `Special1D` band: **[0 -> 0.25]**  (p_of_tot = 0.25)
  - `Special2D` band: **[0.25 -> 1.0]** (p_of_tot = 0.75, stacked on top)
- **Memory axis (-Y), mirrored, stacked from zero downward:**
  Memory is **non-accumulative** (a hierarchy, not a pool), so the axis is split
  into EQUAL 1/N slots (N=2 -> 0.5 each), independent of any peak ratio. The
  slots stack only for layout and are NEVER summed; each is its own 0..1 gauge.
  - `FancyMem` slot: **[0 -> -0.5]**  (p_of_tot = 0.5, nearest zero because
    FancyMem was declared BEFORE DRAM)
  - `DRAM` slot: **[-0.5 -> -1.0]**   (p_of_tot = 0.5, the other equal slot)

So in the captured spec the inline dataset should show, for the memory bands:
EinsumC (memory_type FancyMem) `mr_mem_band_bottom = 0.0`, `mr_mem_band_top =
0.5`; EinsumA / EinsumB (memory_type DRAM) `mr_mem_band_bottom = 0.5`,
`mr_mem_band_top = 1.0`. (If the registry were NOT passed, the legacy fallback
would instead order memory by kernel first-appearance -> DRAM nearest zero,
which is the wrong/old behaviour we are fixing.)

#### How to read each band

- **Each band is capped by a labeled mini-roof line** at its `band_top` (the
  cumulative `p_of_tot` level). ALL roofs are shown, including the cumulative
  outermost one at **+1.0 / -1.0** ("100% of all resources").
- **Colored fill = the Einsum** running on that sub-resource (color = Einsum,
  one hue per kernel, unchanged from every other figure in this notebook); the
  fill rises from the band's fixed `band_bottom`.
- **The white gap between the colored fill and the mini-roof = the
  underutilization** of that sub-resource (head-room left on that band).
- **Solid line** = the kernel's **ideal-if-it-had-100%-of-its-OWN
  sub-resource** time (sub-resource-relative ideal, NOT 100% of all compute /
  memory on the chip).
- **Dashed line** = the **throttled remainder**, drawn as a continuation of
  the solid line (the non-ideal time beyond the solid ideal).
- **A second `strokeDash` border channel encodes the SUB-resource**, with a
  *distinct dash pattern per sub-resource*:
  - on the **positive** (compute, +Y) rects the dash encodes the *compute*
    sub-resource -- `Special1D` (EinsumA) vs `Special2D` (EinsumB, EinsumC);
  - on the **negative** (memory, -Y) rects the dash encodes the *memory*
    sub-resource -- `DRAM` (EinsumA, EinsumB) vs `FancyMem` (EinsumC).
- **A "Resources (share of total)" legend** lists each sub-resource ->
  its dash sample -> its `p_of_tot` pool-share limit:
  - `Special1D = 0.25`  (1D array is 25% of all compute)
  - `Special2D = 0.75`  (2D array is 75% of all compute)
  - `DRAM = 0.5`        (equal 1/N memory slot -- an independent gauge, not a
    share of a combined memory pool)
  - `FancyMem = 0.5`    (equal 1/N memory slot -- likewise an independent gauge)

Because utilization is shown *pool-relative* (`*_raw * p_of_tot`), the colored
fills are short: e.g. EinsumA compute = 0.3 * 0.25 = 0.075 of the whole
compute pool, EinsumC memory = 0.25 * 0.5 = 0.125 read on its own FancyMem slot
gauge (memory slots are independent 0..1 gauges, not shares of a memory pool).


In [15]:
################################################################################
# MULTI-RESOURCE RENDER (the new opt-in visual)
#
# `multi_r_cascade` was built + asserted + tabulated by the cells above and is
# NOT mutated here (CampaignDiagram only reads it), so the earlier assertion
# cell stays valid on a full re-run.
#
# KEY: we pass `resource_registry=reg` into the CampaignDiagram constructor.
# This is what makes the stacked multi-resource bands follow the registry
# DECLARATION order (Special1D, Special2D, FancyMem, DRAM) instead of the
# legacy kernel-appearance fallback. Concretely, with FancyMem declared BEFORE
# DRAM the memory axis stacks FancyMem nearest zero ([0 -> -0.10]) and DRAM
# beyond it ([-0.10 -> -1.0]); the compute axis stacks Special1D ([0 -> 0.25])
# then Special2D ([0.25 -> 1.0]). Without the registry the bands would fall
# back to kernel-appearance order (the wrong/old behaviour we are fixing).
#
# We render the SAME cascade twice, matching this notebook's convention:
#   1. an inline `.show()` of the multi-resource dual view (display_data), then
#   2. a final bare `.interactive()` expression so its Vega-Lite spec is
#      captured as the cell's execute_result (this is the output a reader -- or
#      an automated check -- inspects for the registry-ordered bands, the
#      strokeDash channel, and the "Resources (share of total)" legend).
#
# `multi_resource=True` turns on: a per-SUB-resource strokeDash border on the
# dual-mode area rects (compute sub-resource on the +Y rects, memory
# sub-resource on the -Y rects), the "Resources (share of total)" legend, and
# the +-1.0 "100% of all resources" reference line. It only works with
# dual_mode -- we pass dual_mode=True explicitly even though it auto-enables.
################################################################################

# Wrap the (unmutated) multi-resource cascade in a fresh diagram object,
# handing it the ResourceRegistry so the bands stack in DECLARATION order.
campaign_diagram = CampaignDiagram(multi_r_cascade, resource_registry=reg)

# (1) Inline display of the multi-resource dual view.
campaign_diagram.draw(dual_mode=True, multi_resource=True).interactive().show()

# (2) Final expression -> execute_result carrying the Vega-Lite spec (with the
#     registry-ordered bands + the strokeDash sub-resource channel + the
#     resource-share legend).
campaign_diagram.draw(dual_mode=True, multi_resource=True).interactive()


alt.LayerChart(...)

alt.LayerChart(...)

### Multi-resource view after tiling + pipelining + throttling

The same multi-resource encoding survives the scheduling transforms. Below we
tile each kernel into 4, pipeline across 3 stages, and throttle to fit the
resource constraints, then render with `dual_mode=True, multi_resource=True`.

To keep the earlier assertion / table cells valid on a full notebook re-run,
we build a **fresh** cascade here from freshly-constructed kernels (registered
against the same `reg`) rather than transforming the shared `multi_r_cascade`
object the assertions depend on. The expected reading is identical to the
figure above (color = Einsum, strokeDash = sub-resource, solid = own-sub-
resource ideal, dashed = remainder, +-1.0 = 100% of all resources) -- just
spread across the tiled / pipelined / throttled schedule.


In [16]:
################################################################################
# MULTI-RESOURCE RENDER  --  tiled(4) + pipelined(3) + throttled
#
# tile()/pipeline()/throttle() each return NEW Cascade objects, but to be
# defensive (and so a full re-run never perturbs the asserted `multi_r_cascade`)
# we construct an INDEPENDENT cascade from fresh kernels here, register them
# against the SAME `reg` registry, then apply the schedule transforms to that
# throwaway copy only.
#
# As with the base render above, we pass `resource_registry=reg` into the
# CampaignDiagram constructor so the stacked bands follow the registry
# DECLARATION order (Special1D, Special2D, FancyMem, DRAM) -- FancyMem nearest
# zero on the memory axis ([0 -> -0.10]), DRAM beyond it ([-0.10 -> -1.0]) --
# even after tiling / pipelining / throttling spreads the schedule out.
################################################################################

# --- Fresh kernels: identical numbers to the asserted scenario ---------------
# EinsumA: 30% of the Special1D compute array, 15% of DRAM bandwidth.
mr_k_a = Kernel(name='EinsumA',
                start=0,
                duration=5,
                compute_util=0.3,            # 30% of Special1D
                bw_util=0.15,                # 15% of DRAM
                compute_type="Special1D",
                memory_type="DRAM")

# EinsumB: 20% of the Special2D compute array, 60% of DRAM bandwidth.
mr_k_b = Kernel(name='EinsumB',
                start=mr_k_a.end,            # chain: C dep B dep A
                duration=10,
                compute_util=0.2,            # 20% of Special2D
                bw_util=0.6,                 # 60% of DRAM
                compute_type="Special2D",
                memory_type="DRAM")

# EinsumC: 50% of the Special2D compute array, 25% of the FancyMem bandwidth.
mr_k_c = Kernel(name='EinsumC',
                start=mr_k_b.end,
                duration=5,
                compute_util=0.5,            # 50% of Special2D
                bw_util=0.25,                # 25% of FancyMem
                compute_type="Special2D",
                memory_type="FancyMem")

# Recompute each kernel's pool-relative view from the SAME registry `reg`
# (Special1D=0.25, Special2D=0.75, FancyMem=0.1, DRAM=0.9) defined earlier.
for _mr_k in (mr_k_a, mr_k_b, mr_k_c):
    _mr_k.update_global_util_view(registry=reg)

# Independent cascade -- the asserted `multi_r_cascade` is left untouched.
mr_tiled_cascade = Cascade(name="Three-step Cascade (multi-resource)",
                           kernels=[mr_k_a, mr_k_b, mr_k_c])

# --- Schedule transforms ------------------------------------------------------
#   .tile(4)     : split every kernel into 4 equal-duration tiles
#   .pipeline(3) : overlap across 3 pipeline stages
#   .throttle()  : slow tiles down so concurrent resource use stays in budget
mr_scheduled_cascade = mr_tiled_cascade.tile(4).pipeline(3).throttle()

# Wrap the scheduled cascade in a diagram object, again handing it the SAME
# ResourceRegistry so the bands stack in DECLARATION order (not the legacy
# kernel-appearance fallback).
campaign_diagram = CampaignDiagram(mr_scheduled_cascade, resource_registry=reg)

# (1) Inline display of the tiled/pipelined/throttled multi-resource view.
campaign_diagram.draw(dual_mode=True, multi_resource=True).interactive().show()

# (2) Final expression -> execute_result carrying the Vega-Lite spec (with the
#     registry-ordered bands + the strokeDash sub-resource channel + the
#     resource-share legend).
campaign_diagram.draw(dual_mode=True, multi_resource=True).interactive()


alt.LayerChart(...)

alt.LayerChart(...)

## Colocated mode (`dual_mode=False`) — multi-resource

This is the **colocated** analogue of the dual multi-resource figures above:
compute is drawn *up* and memory is treated as the *"pipe"* underneath it, all
on a **single Y axis** (no mirrored +Y / -Y split).

- **Compute sub-resources** appear as full-width **dark-grey boundary lines**
  at their *cumulative* `p_of_tot` level, in **registry-declaration order**
  (Special1D, Special2D): `Special1D → 0.25`, then `Special2D → 0.25 + 0.75 =
  **1.00**`. Each boundary line is labeled with its sub-resource name.
- **The Einsum compute step-line is OFFSET** by its compute sub-resource's
  band *bottom* (exactly like dual): a `Special2D` Einsum starts from where
  `Special1D` ended. So `EinsumA` (Special1D) starts at `0`; `EinsumB`
  (Special2D) starts at `0.25` → `0.25 + 0.15 = **0.40**`; `EinsumC`
  (Special2D) starts at `0.25` → `0.25 + 0.375 = **0.625**`.
- **Memory keeps the single-resource "pipe"**, but the grey is now
  **partitioned per memory sub-resource** by `p_of_tot` (an independent grey
  ramp, FancyMem then DRAM in declaration order). Each Einsum's colored
  used-bandwidth fill is **offset to start at the bottom of its memory
  sub-resource's segment** — e.g. a `DRAM` Einsum's fill starts at FancyMem's
  cumulative `0.10`, not at `0`.
- The **same combined "Resources" legend** appears on the right, identical to
  the dual figures.


In [20]:
################################################################################
# COLOCATED MULTI-RESOURCE RENDER  --  base (dual_mode=False)
#
# This is the COLOCATED counterpart of the dual base render above. We reuse the
# SAME already-built `multi_r_cascade` and the SAME `reg` ResourceRegistry that
# the dual base cell used -- we do NOT rebuild or re-bind anything here, we just
# render that identical (unmutated) cascade the colocated way. CampaignDiagram
# only READS the cascade, so the earlier assertion / table cells stay valid on
# a full notebook re-run.
#
# As in the dual base cell, we pass `resource_registry=reg` into the
# CampaignDiagram CONSTRUCTOR so the per-sub-resource boundaries follow the
# registry DECLARATION order (Special1D, Special2D, FancyMem, DRAM) rather than
# the legacy kernel-appearance fallback. Concretely, on the single compute axis
# the dark-grey compute boundary lines land at the CUMULATIVE p_of_tot levels
# Special1D -> 0.25 then Special2D -> 1.00, and each Einsum's compute step-line
# is offset by its compute sub-resource's band bottom (EinsumB/EinsumC, both
# Special2D, start from 0.25). The memory "pipe" grey is partitioned per memory
# sub-resource (FancyMem then DRAM) and each Einsum's used-bandwidth fill is
# offset to the bottom of its memory sub-resource segment.
#
# The ONLY difference from the dual base cell is `dual_mode=False` (colocated:
# compute up, memory as the pipe, single Y axis) instead of `dual_mode=True`.
#
# We render the SAME cascade twice, matching this notebook's convention:
#   1. an inline `.show()` of the colocated multi-resource view (display_data),
#   2. a final bare `.interactive()` expression so its Vega-Lite spec is
#      captured as the cell's execute_result.
################################################################################

# Wrap the (unmutated) shared multi-resource cascade in a fresh diagram object,
# handing it the SAME ResourceRegistry so the boundaries follow DECLARATION
# order. Note we reuse `multi_r_cascade` and `reg` exactly as built earlier.
campaign_diagram = CampaignDiagram(multi_r_cascade, resource_registry=reg)

# (1) Inline display of the colocated multi-resource view.
campaign_diagram.draw(dual_mode=False, multi_resource=True).interactive().show()

# (2) Final expression -> execute_result carrying the Vega-Lite spec for the
#     colocated multi-resource view (per-sub-resource compute boundary lines,
#     partitioned memory-pipe grey, combined "Resources" legend on the right).
campaign_diagram.draw(dual_mode=True, multi_resource=True).interactive().show()


alt.LayerChart(...)

alt.LayerChart(...)

Colocated multi-resource with tiling + pipelining + throttling:

In [21]:
################################################################################
# COLOCATED MULTI-RESOURCE RENDER  --  tiled(4) + pipelined(3) + throttled
#
# COLOCATED counterpart of the dual tiled/pipelined/throttled render above.
# tile()/pipeline()/throttle() each return NEW Cascade objects, but to be
# defensive (and so a full re-run never perturbs the asserted `multi_r_cascade`)
# we construct an INDEPENDENT cascade from fresh kernels here -- EXACTLY as the
# dual tiled cell does -- register them against the SAME `reg` registry, then
# apply the schedule transforms to that throwaway copy only.
#
# The fresh kernels, the registry, and the .tile(4).pipeline(3).throttle()
# transform chain are byte-for-byte the same construction as the dual tiled
# cell. The ONLY rendering difference is the .draw(...) call: colocated
# (`dual_mode=False`) and we also pass `resource_registry=reg` explicitly into
# draw() (the same registry object the constructor already received) so the
# colocated multi-resource boundaries follow the registry DECLARATION order
# (Special1D, Special2D, FancyMem, DRAM) even after the schedule spreads out.
################################################################################

# --- Fresh kernels: identical numbers to the asserted scenario ---------------
# EinsumA: 30% of the Special1D compute array, 15% of DRAM bandwidth.
mr_k_a = Kernel(name='EinsumA',
                start=0,
                duration=5,
                compute_util=0.3,            # 30% of Special1D
                bw_util=0.15,                # 15% of DRAM
                compute_type="Special1D",
                memory_type="DRAM")

# EinsumB: 20% of the Special2D compute array, 60% of DRAM bandwidth.
mr_k_b = Kernel(name='EinsumB',
                start=mr_k_a.end,            # chain: C dep B dep A
                duration=10,
                compute_util=0.2,            # 20% of Special2D
                bw_util=0.6,                 # 60% of DRAM
                compute_type="Special2D",
                memory_type="DRAM")

# EinsumC: 50% of the Special2D compute array, 25% of the FancyMem bandwidth.
mr_k_c = Kernel(name='EinsumC',
                start=mr_k_b.end,
                duration=5,
                compute_util=0.5,            # 50% of Special2D
                bw_util=0.25,                # 25% of FancyMem
                compute_type="Special2D",
                memory_type="FancyMem")

# Recompute each kernel's pool-relative view from the SAME registry `reg`
# (Special1D=0.25, Special2D=0.75, FancyMem=0.1, DRAM=0.9) defined earlier.
for _mr_k in (mr_k_a, mr_k_b, mr_k_c):
    _mr_k.update_global_util_view(registry=reg)

# Independent cascade -- the asserted `multi_r_cascade` is left untouched.
mr_tiled_cascade = Cascade(name="Three-step Cascade (multi-resource)",
                           kernels=[mr_k_a, mr_k_b, mr_k_c])

# --- Schedule transforms ------------------------------------------------------
#   .tile(4)     : split every kernel into 4 equal-duration tiles
#   .pipeline(3) : overlap across 3 pipeline stages
#   .throttle()  : slow tiles down so concurrent resource use stays in budget
mr_scheduled_cascade = mr_tiled_cascade.tile(4).pipeline(3).throttle()

# Wrap the scheduled cascade in a diagram object, again handing it the SAME
# ResourceRegistry so the boundaries stack in DECLARATION order (not the legacy
# kernel-appearance fallback).
campaign_diagram = CampaignDiagram(mr_scheduled_cascade, resource_registry=reg)

# (1) Inline display of the tiled/pipelined/throttled COLOCATED multi-resource
#     view. Colocated (`dual_mode=False`); `resource_registry=reg` passed
#     explicitly to draw() as well so the band ordering is unambiguous.
campaign_diagram.draw(dual_mode=False, multi_resource=True, resource_registry=reg).interactive().show()

# (2) Final expression -> execute_result carrying the Vega-Lite spec for the
#     colocated tiled/pipelined/throttled multi-resource view.
campaign_diagram.draw(dual_mode=True, multi_resource=True, resource_registry=reg).interactive().show()


alt.LayerChart(...)

alt.LayerChart(...)

- This should plot, for each, its resource utilization for the TOTAL (so 0.3*0.25)
- It should have a solid line for the ideal time (actual_time/(1/spec_resource_frac))
- and a dashed line for the remaining
- the ideal time is doen relative to it's sub-resource, not the total resource 
-    (if I use 50% of 1D array, ideal time should be if I could use 100% of the 1D array, 
-     not 100% of all compute present)
- this is the MAX of (comp_ideal, mem_ideal)
- dashed lines now mean :
  - throttled duration
  - dilated duration
  - non-ideal runtime over ideal runtime -- (should I change this to *per resource*?)


In [19]:
# A campaign diagram is 
kernel4a = Kernel(name='EinsumA',
                  start=0,
                  duration=5,
                  compute_util=0.3,
                  bw_util=0.15, 
                  compute_type = "Special1D",
                  memory_type = "DRAM"
                 )


kernel4b = Kernel(name='EinsumB',
                  start=kernel4a.end,
                  duration=10,
                  compute_util=0.2,
                  bw_util=0.6,
                  compute_type = "Special2D",
                  memory_type = "DRAM"
                 )


kernel4c = Kernel(name='EinsumC',
                  start=kernel4b.end,
                  duration=5,
                  compute_util=0.5,   # was 0.2
                  bw_util=0.25,
                  compute_type = "Special2D",
                  memory_type = "FancyMem"
                 )

![image](SCRIBBLES_2025.jpeg)

## Worked example: the `accumulative` flag end to end

This self-contained section demonstrates BOTH regimes side by side, driven only
by the flag (no hand-set memory shares):

- **Compute (accumulative, peak-share).** Two compute engines with real peak
  throughputs — a Tensor core (`peak=125.0`) and an FMA/CUDA-core path
  (`peak=15.7`). `derive_p_of_tot()` turns those into pooled band heights
  `peak / Σpeak` → Tensor ≈ **0.888**, FMA ≈ **0.112**, which **sum to 1.0**
  (the combined compute pool).

- **Memory (non-accumulative, equal slots).** Two memory levels — `L2` and
  `DRAM`. Their peaks are irrelevant to the height: each gets an equal `1/N`
  slot → **0.5 / 0.5**, an independent 0..1 gauge. Below, only `DRAM` is bound
  by kernels; `L2` is declared but unused, so it renders as an **empty 0.5 slot**
  rather than collapsing — a level that a phase does not touch still shows its
  head-room.

In [ ]:
################################################################################
# accumulative-flag worked example: compute peak-share vs memory equal-slots.
################################################################################
demo_reg = ResourceRegistry()

# Compute: ACCUMULATIVE (default). Give real peaks -> derive_p_of_tot() sets the
# heights to peak-shares that POOL (sum to 1.0).
demo_reg.add_compute("Tensor", peak=125.0)
demo_reg.add_compute("FMA",    peak=15.7)

# Memory: NON-ACCUMULATIVE (default). No peaks needed for the height -- the axis
# splits into equal 1/N slots. L2 is declared but left UNBOUND on purpose to
# show that an empty memory slot is still drawn (an independent gauge at 0).
demo_reg.add_memory("L2")
demo_reg.add_memory("DRAM")

# One call sets every band height from the flags.
demo_reg.derive_p_of_tot()

print("derived band heights (p_of_tot):")
for _n in ("Tensor", "FMA", "L2", "DRAM"):
    _a = demo_reg.get(_n).attrs
    print(f"  {_n:7s} accumulative={_a['accumulative']!s:5s} "
          f"p_of_tot={_a['p_of_tot']:.6g}")
print(f"  compute peak-shares sum to "
      f"{demo_reg.get('Tensor').attrs['p_of_tot'] + demo_reg.get('FMA').attrs['p_of_tot']:.6g} "
      f"(pooled); memory slots are equal 0.5/0.5 (independent gauges).")

# Two kernels, both on DRAM (L2 stays an empty slot); one per compute engine.
demo_k_gemm = Kernel(name="GEMM", start=0, duration=6,
                     compute_util=0.9, bw_util=0.4,
                     compute_type="Tensor", memory_type="DRAM")
demo_k_attn = Kernel(name="Attention", start=demo_k_gemm.end, duration=6,
                     compute_util=0.7, bw_util=0.6,
                     compute_type="FMA", memory_type="DRAM")
for _k in (demo_k_gemm, demo_k_attn):
    _k.update_global_util_view(registry=demo_reg)

demo_cascade = Cascade(name="Accumulative-flag demo",
                       kernels=[demo_k_gemm, demo_k_attn])

# Dual multi-resource render: compute peak-share bands up top (Tensor tall, FMA
# short), memory equal 0.5 slots below (DRAM filled, L2 empty but drawn).
CampaignDiagram(demo_cascade, resource_registry=demo_reg).draw(
    dual_mode=True, multi_resource=True).interactive()